# Setup packages

In [1]:
import omicstl
import pandas as pd
import numpy as np
from pathlib import Path

# Bring in Data


In [2]:
from pathlib import Path

repo_root = Path("/workspaces/timed-hpc")
data_dir = repo_root / "viral_use_case" / "data"
out_dir = repo_root / "viral_use_case" / "model_outputs"
out_dir.mkdir(parents=True, exist_ok=True)

source_path = data_dir / "source_dset.csv"
target_transfer_path = data_dir / "target_transfer.csv"
target_validation_path = data_dir / "target_validation.csv"

print("Using files:")
print(source_path)
print(target_transfer_path)
print(target_validation_path)

Using files:
/workspaces/timed-hpc/viral_use_case/data/source_dset.csv
/workspaces/timed-hpc/viral_use_case/data/target_transfer.csv
/workspaces/timed-hpc/viral_use_case/data/target_validation.csv


In [3]:
# Read data
# =========================================
base_source_data = pd.read_csv(source_path).set_index("SampleID")
base_target_transfer_data = pd.read_csv(target_transfer_path).set_index("SampleID")
base_target_validation_data = pd.read_csv(target_validation_path).set_index("SampleID")

for name, df in {
    "source": base_source_data,
    "target_transfer": base_target_transfer_data,
    "target_validation": base_target_validation_data,
}.items():
    if "Resp" not in df.columns:
        raise ValueError(f"{name} is missing 'Resp'")
    df["Resp"] = df["Resp"].apply(lambda x: 2 if x == "viral" else 1)

print("\nShapes:")
print("source:", base_source_data.shape)
print("target_transfer:", base_target_transfer_data.shape)
print("target_validation:", base_target_validation_data.shape)

print("\nResponse counts:")
print(base_source_data["Resp"].value_counts(dropna=False))
print(base_target_transfer_data["Resp"].value_counts(dropna=False))
print(base_target_validation_data["Resp"].value_counts(dropna=False))


Shapes:
source: (358, 476)
target_transfer: (192, 476)
target_validation: (48, 476)

Response counts:
Resp
1    179
2    179
Name: count, dtype: int64
Resp
2    147
1     45
Name: count, dtype: int64
Resp
2    33
1    15
Name: count, dtype: int64


# Setup Data Container

In [4]:
# =========================================
#  Dataset container
# =========================================
from omicstl.simulation_utils.data_utils import DatasetContainer
all_datasets = DatasetContainer(
    source_data=base_source_data,
    target_data=base_target_transfer_data,
    target_test_data=[base_target_validation_data],
)
all_datasets.set_response_column("Resp")

source_input = all_datasets.source_data.drop(columns=["Resp"]).copy()
target_input = all_datasets.target_test_data[0].drop(columns=["Resp"]).copy()
target_truth = all_datasets.target_test_data[0]["Resp"].copy()

feature_names = source_input.columns.tolist()

print("\nFeature matrix shapes:")
print("source_input:", source_input.shape)
print("target_input:", target_input.shape)



Feature matrix shapes:
source_input: (358, 475)
target_input: (48, 475)


# Model Fitting

In [5]:
# 4. Params
# =========================================
param_grid = {
    "dropout": [0.25, 0.5],
    "n_latent_dims": [2],
    "hidden_dim_base": [6],
    "lr": [0.01, 0.001],
    "source_epochs": [1000],
    "target_epochs": [1000],
    "freeze": ["none"],
    "weight_decay": [1e-4, 1e-2],
    "gamma": [1, 2, 3],
}


In [6]:
# =========================================
#  Fit models
# =========================================
import torch
from torch import device
import random
from omicstl.simulation_utils.model_utils import fit_dl_model, fit_rf_model
random.seed(1123)
torch.manual_seed(42)

mlp_out = mlp_model = mlp_model_targetonly = None
vae_out = vae_model = vae_model_targetonly = None
rf_out = rf_model = None

try:
    mlp_out, mlp_model, mlp_model_targetonly = fit_dl_model(
        all_datasets,
        "mult_mlp",
        device("cpu"),
        param_grid,
    )
    print("\nMLP fit complete")
except Exception as e:
    print("\nFailed MLP")
    print(e)

torch.manual_seed(42)
try:
    vae_out, vae_model, vae_model_targetonly = fit_dl_model(
        all_datasets,
        "mult_vae",
        device("cpu"),
        param_grid,
    )
    print("\nVAE fit complete")
except Exception as e:
    print("\nFailed VAE")
    print(e)

random.seed(42)
try:
    rf_out, rf_model = fit_rf_model(all_datasets)
    print("\nRF fit complete")
except Exception as e:
    print("\nFailed RF")
    print(e)

/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)



MLP fit complete


/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)



VAE fit complete

RF fit complete


/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)


In [7]:
display(mlp_out)
display(vae_out)
display(rf_out)

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,NaN,NaN,0.854167,0.904110,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.01,2.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,NaN,NaN,0.500000,0.538462,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.01,3.0


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,NaN,NaN,0.8125,0.873239,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0100,2.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,NaN,NaN,0.3125,0.000000,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.0001,1.0


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall,roc_auc
0,None,None,test_0,rf,pred_source_full,target,NaN,NaN,0.312500,0.000000,0.000000,0.000000,0.000000,0.310101
1,None,None,test_0,rf,pred_source_full_val,target,NaN,NaN,0.312500,0.000000,0.000000,0.000000,0.000000,0.310101
2,None,None,test_0,rf,pred_0_full,target,NaN,NaN,0.541667,0.592593,0.141569,0.761905,0.484848,0.777778
3,None,None,test_0,rf,pred_0_full_val,target,NaN,NaN,0.479167,0.509804,0.058026,0.722222,0.393939,0.777778
4,None,None,test_0,rf,pred_1_full,target,NaN,NaN,0.583333,0.629630,0.232173,0.809524,0.515152,0.771717
5,None,None,test_0,rf,pred_1_full_val,target,NaN,NaN,0.520833,0.581818,0.078931,0.727273,0.484848,0.771717
6,None,None,test_0,rf,pred_2_full,target,NaN,NaN,0.583333,0.629630,0.232173,0.809524,0.515152,0.773737
7,None,None,test_0,rf,pred_2_full_val,target,NaN,NaN,0.437500,0.470588,-0.034816,0.666667,0.363636,0.773737
8,None,None,test_0,rf,pred_3_full,target,NaN,NaN,0.479167,0.528302,0.022792,0.700000,0.424242,0.789899
9,None,None,test_0,rf,pred_3_full_val,target,NaN,NaN,0.562500,0.618182,0.169138,0.772727,0.515152,0.789899
